# Task 2: Laning & Overtaking with SB3 PPO

This notebook trains a PPO agent on the newer `highway-env` / Stable-Baselines3 API, following the older Task 2 setup from `racetrack-agents` where three slower non-agent vehicles are spawned and the ego vehicle must lane-follow while overtaking.

The older DQN command used `--spawn_vehicles 3`, `--batch_size 256`, `--lr 0.00005`, `--lr_decay`, `--arch Identity`, and `--fc_layers 3`. The cells below map those ideas to SB3 PPO with a 3-layer MLP policy, linear learning-rate decay, and `other_vehicles=3` in the `racetrack-oval-v0` config.

In [1]:
# If this notebook is running in a fresh environment, install the core packages first.
# In the local repo environment you can usually leave this cell commented out.
#
# %pip install "highway-env>=1.8" "stable-baselines3[extra]>=2.0" tensorboard moviepy

from pathlib import Path
import base64
import os
import random

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import highway_env  # Registers highway-env environments in many versions.
import numpy as np
import torch

from IPython.display import HTML, display
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv

# Newer gymnasium versions can register an external environment package explicitly.
# Older highway-env versions register on import, so we keep this tolerant.
try:
    gym.register_envs(highway_env)
except Exception:
    pass


c:\Users\16469\anaconda3\envs\circuit\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


## Experiment Config

Task 2 is represented by `other_vehicles=3`. The rest of the environment config follows `sb3_racetrack_oval_ppo.py`, with a richer occupancy-grid observation and lateral-only continuous control.

In [2]:
SEED = 42
ENV_ID = "racetrack-oval-v0"

# Keep full training as the default. For an end-to-end notebook smoke test, run
# `FAST_DEV_RUN=1` in the process environment before executing the notebook.
FAST_DEV_RUN = os.environ.get("FAST_DEV_RUN", "0") == "1"
EXP_ID = "sb3_ppo_task2_laning_overtaking_fastdev" if FAST_DEV_RUN else "sb3_ppo_task2_laning_overtaking"

CWD = Path.cwd()
if (CWD / "racetrack_env.py").exists():
    WORK_DIR = CWD
elif (CWD / "racetrack-agents").exists():
    WORK_DIR = CWD / "racetrack-agents"
else:
    WORK_DIR = CWD

RUN_DIR = WORK_DIR / "runs" / EXP_ID
MODEL_DIR = RUN_DIR / "models"
BEST_MODEL_DIR = MODEL_DIR / "best"
CHECKPOINT_DIR = MODEL_DIR / "checkpoints"
LOG_DIR = RUN_DIR / "logs"
VIDEO_DIR = RUN_DIR / "videos"

for directory in [MODEL_DIR, BEST_MODEL_DIR, CHECKPOINT_DIR, LOG_DIR, VIDEO_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Reproducibility: exact runs can still vary across machines/GPU kernels.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Mapping from the older DQN command:
#   --spawn_vehicles 3  -> other_vehicles=3
#   --batch_size 256    -> PPO minibatch size 256
#   --lr 0.00005        -> PPO starting learning rate 5e-5
#   --lr_decay          -> linear schedule to 0
#   --arch Identity     -> flatten the occupancy grid, then use an MLP
#   --fc_layers 3       -> 3 hidden layers in the actor and critic
TASK2_CONFIG = {
    "observation": {
        "type": "OccupancyGrid",
        "features": [
            "presence",
            "on_road",
            "x",
            "y",
            "vx",
            "vy",
            "cos_h",
            "sin_h",
            "long_off",
            "lat_off",
            "ang_off",
        ],
        "grid_size": [[-18, 18], [-18, 18]],
        "grid_step": [3, 3],
        "as_image": False,
        "align_to_vehicle_axes": True,
    },
    "action": {
        "type": "ContinuousAction",
        "longitudinal": False,
        "lateral": True,
        "target_speeds": [0, 5, 10],
    },
    "simulation_frequency": 15,
    "policy_frequency": 5,
    "duration": 120,
    "collision_reward": -1000,
    "lane_centering_cost": 4,
    "lane_centering_reward": 1,
    "action_reward": -100,
    "controlled_vehicles": 1,
    "other_vehicles": 3,
    "screen_width": 1000,
    "screen_height": 1000,
    "centering_position": [0.5, 0.5],
    "speed_limit": 10.0,
    "terminate_off_road": True,
    "length": 100,
    "no_lanes": 3,
}

# 5000 older episodes * 120-step newer episode horizon.
# Increase this for stronger policies; reduce it for a quick smoke test.
N_EPISODES = 1 if FAST_DEV_RUN else 5000
TOTAL_TIMESTEPS = 16 if FAST_DEV_RUN else N_EPISODES * TASK2_CONFIG["duration"]

N_ENVS = 1 if FAST_DEV_RUN else min(8, max(1, os.cpu_count() or 1))
BATCH_SIZE = 8 if FAST_DEV_RUN else 256
N_STEPS = 8 if FAST_DEV_RUN else 2048
LEARNING_RATE = 5e-5
EVAL_FREQ = 8 if FAST_DEV_RUN else max(10_000 // N_ENVS, 1)
N_EVAL_EPISODES = 1 if FAST_DEV_RUN else 5


## Build Training and Evaluation Environments

SB3 trains on vectorized environments. `DummyVecEnv` is the safest default inside notebooks on Windows. For longer command-line runs, set `USE_SUBPROC = True`.

In [3]:
USE_SUBPROC = False

def make_task2_env(render_mode=None):
    """Create one Task 2 racetrack environment."""
    return gym.make(ENV_ID, config=TASK2_CONFIG, render_mode=render_mode)

vec_env_cls = SubprocVecEnv if USE_SUBPROC and N_ENVS > 1 else DummyVecEnv

train_env = make_vec_env(
    lambda: make_task2_env(),
    n_envs=N_ENVS,
    seed=SEED,
    vec_env_cls=vec_env_cls,
)
# Evaluation stays single-env so callback results are easy to interpret.
eval_env = make_vec_env(
    lambda: make_task2_env(),
    n_envs=1,
    seed=SEED + 10_000,
    vec_env_cls=DummyVecEnv,
)


## Define PPO

The PPO policy uses a 3-layer actor and critic MLP, matching the spirit of `--arch Identity --fc_layers 3` from the older code: flatten the occupancy grid, then learn dense policy/value heads.

In [4]:
def linear_schedule(initial_value):
    """SB3 schedule: progress_remaining moves from 1.0 to 0.0 during training."""
    def schedule(progress_remaining):
        return progress_remaining * initial_value
    return schedule

policy_kwargs = {
    "activation_fn": torch.nn.Tanh,
    "net_arch": {
        "pi": [256, 256, 256],
        "vf": [256, 256, 256],
    },
}

model = PPO(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=linear_schedule(LEARNING_RATE),
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=10,
    gamma=0.90,
    gae_lambda=0.95,
    clip_range=0.20,
    ent_coef=0.001,
    max_grad_norm=0.5,
    policy_kwargs=policy_kwargs,
    tensorboard_log=str(LOG_DIR),
    seed=SEED,
    verbose=1,
)

model.policy


Using cuda device


ActorCriticPolicy(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (pi_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (vf_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mlp_extractor): MlpExtractor(
    (policy_net): Sequential(
      (0): Linear(in_features=1584, out_features=256, bias=True)
      (1): Tanh()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Tanh()
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): Tanh()
    )
    (value_net): Sequential(
      (0): Linear(in_features=1584, out_features=256, bias=True)
      (1): Tanh()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): Tanh()
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): Tanh()
    )
  )
  (action_net): Linear(in_features=256, out_features=1, bias=True)
  (value_net): Linear(in

## Train and Save the Best Model

`EvalCallback` periodically runs deterministic evaluations and writes the best model to disk. TensorBoard logs are stored under `runs/sb3_ppo_task2_laning_overtaking/logs`.

In [5]:
%load_ext tensorboard
%tensorboard --logdir runs/sb3_ppo_task2_laning_overtaking/logs

In [ ]:
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=str(BEST_MODEL_DIR),
    log_path=str(LOG_DIR / "eval"),
    eval_freq=EVAL_FREQ,
    n_eval_episodes=N_EVAL_EPISODES,
    deterministic=True,
    render=False,
)

checkpoint_callback = CheckpointCallback(
    save_freq=max(50_000 // N_ENVS, 1),
    save_path=str(CHECKPOINT_DIR),
    name_prefix="ppo_task2_checkpoint",
)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[eval_callback, checkpoint_callback],
    tb_log_name="PPO_task2",
)

model.save(MODEL_DIR / "ppo_task2_last")
train_env.close()
eval_env.close()


Logging to c:\Users\16469\Desktop\circuit\racetrack-agents\runs\sb3_ppo_task2_laning_overtaking\logs\PPO_task2_4
Eval num_timesteps=10000, episode_reward=42.59 +/- 7.00
Episode length: 43.60 +/- 7.00
---------------------------------
| eval/              |          |
|    mean_ep_length  | 43.6     |
|    mean_reward     | 42.6     |
| time/              |          |
|    total_timesteps | 10000    |
---------------------------------
New best mean reward!
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 13.2     |
|    ep_rew_mean     | 11.4     |
| time/              |          |
|    fps             | 29       |
|    iterations      | 1        |
|    time_elapsed    | 551      |
|    total_timesteps | 16384    |
---------------------------------
Eval num_timesteps=20000, episode_reward=234.38 +/- 150.05
Episode length: 238.40 +/- 151.97
-----------------------------------------
| eval/                   |             |
|    mean_ep_length    

## Load the Best PPO Checkpoint

If training was interrupted before an evaluation improved, fall back to the last saved model.

In [ ]:
best_model_path = BEST_MODEL_DIR / "best_model.zip"
last_model_path = MODEL_DIR / "ppo_task2_last.zip"

if best_model_path.exists():
    trained_model = PPO.load(best_model_path)
    print(f"Loaded best model: {best_model_path}")
else:
    trained_model = PPO.load(last_model_path)
    print(f"Best model was not found, loaded last model: {last_model_path}")


## Find and Export the Best Evaluation Episode

The first pass evaluates deterministic rollouts over fixed seeds without recording. The best seed is then replayed once with `RecordVideo`, producing a single video for the strongest episode found in this sweep.

In [ ]:
def run_episode(model, seed, record=False, name_prefix="ppo_task2_best_episode"):
    """Run one deterministic episode and optionally record it to VIDEO_DIR."""
    render_mode = "rgb_array" if record else None
    env = gym.make(ENV_ID, config=TASK2_CONFIG, render_mode=render_mode)

    if record:
        env = RecordVideo(
            env,
            video_folder=str(VIDEO_DIR),
            name_prefix=name_prefix,
            episode_trigger=lambda episode_id: episode_id == 0,
        )
    obs, info = env.reset(seed=seed)
    done = False
    truncated = False
    total_reward = 0.0
    episode_length = 0

    while not (done or truncated):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        total_reward += float(reward)
        episode_length += 1
        if record:
            env.render()

    env.close()
    return total_reward, episode_length

candidate_seeds = list(range(SEED, SEED + (1 if FAST_DEV_RUN else 25)))
episode_scores = []

for seed in candidate_seeds:
    reward, length = run_episode(trained_model, seed=seed, record=False)
    episode_scores.append({"seed": seed, "reward": reward, "length": length})

best_episode = max(episode_scores, key=lambda item: item["reward"])
best_episode


In [ ]:
video_prefix = f"ppo_task2_best_seed_{best_episode['seed']}"
recorded_reward, recorded_length = run_episode(
    trained_model,
    seed=best_episode["seed"],
    record=True,
    name_prefix=video_prefix,
)

video_files = sorted(VIDEO_DIR.glob(f"{video_prefix}*.mp4"), key=lambda path: path.stat().st_mtime)
best_video_path = video_files[-1] if video_files else None

print(f"Recorded reward: {recorded_reward:.3f}")
print(f"Recorded length: {recorded_length}")
print(f"Video path: {best_video_path}")


## Display the Exported Video

In [ ]:
def show_video(video_path, width=720):
    """Embed an exported mp4 directly in the notebook."""
    video_path = Path(video_path)
    video_bytes = video_path.read_bytes()
    encoded = base64.b64encode(video_bytes).decode("ascii")
    display(HTML(f"""
    <video width="{width}" controls>
      <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
    </video>
    """))

if best_video_path is not None:
    show_video(best_video_path)
else:
    print("No video file was found. Check that moviepy/ffmpeg are installed and rerun the recording cell.")


## Optional: TensorBoard

Run this cell while training or after training to inspect reward, loss, entropy, KL, and evaluation curves.

In [ ]:
# Uncomment these lines in an interactive notebook session.

